# different engines

Yosys supports a number of different solver engines. You can find a list of them on the .sby [reference site](https://symbiyosys.readthedocs.io/en/latest/reference.html).

The `YosysTestCase` class directly forwards the `engine` argument of `_yosys_params_` to the .sby project file. Other than that, the parameter has no effect. Consequently, all engines should also work with the cohdl_yosys library.

This notebook contains a brief overview of all engines contained in the cohdl docker image.

In [1]:
# basic setup of jupyter notebook

import cohdl
from cohdl import Bit, Unsigned, Port
from cohdl_yosys import YosysParams, YosysTestCase, formal

from cohdl_yosys.formal import (
    When,
    always,
    cover,
    prev,
    stable
)

import cohdl.std as std

cohdl.use_pretty_traceback(False)

## example setup

In [2]:
class ExampleEntity(cohdl.Entity):
    clk = Port.input(Bit)

    enable = Port.input(Bit)
    input = Port.input(Unsigned[8])

    result = Port.output(Unsigned[16], default=0)

    def architecture(self):
        
        @std.sequential(std.Clock(self.clk))
        def proc_accumulate():
            if self.enable:
                self.result <<= self.result + self.input

class BaseTestCase(YosysTestCase):
    _yosys_params_ = YosysParams(use_tmp_dir=True, quiet=True)

    # used for positive/negative test
    fail_check = False

    def formal_properties(self, dut):
        pass

    def architecture(self, dut: ExampleEntity):
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():
            self.formal_properties(dut)

def run_check(TestCase: type[BaseTestCase], engine):
    print(f"check engine={engine}")

    TestCase.fail_check = False
    positive_result =  TestCase().test_formal_properties(return_on_error=True)
    TestCase.fail_check = True
    negative_result = not TestCase().test_formal_properties(return_on_error=True)

    print("  positive test: ", ("FAILED", "PASSED")[positive_result])
    print("  negative test: ", ("FAILED", "PASSED")[negative_result])

## solvers for bmc mode

In [3]:
def check_bmc_mode():

    not_working_engines = (
        "abc sim3", # deadlocks?
    )

    working_engines = (
        None,
        "smtbmc",
        "btor btormc",
        "btor pono",
        "abc bmc3",
        "aiger aigbmc",
    )

    for engine in working_engines:

        class Check_BMC(BaseTestCase, entity=ExampleEntity):
            _yosys_params_ = YosysParams(bmc=True, engines=engine)

            def formal_properties(self, dut: ExampleEntity):
                always["stable_when_not_en"](When(not dut.enable).then_next(stable(dut.result)))
                always["increments_when_en"](When(dut.enable).then_next(dut.result == prev(dut.result) + prev(dut.input)))

                if not self.fail_check:
                    # limit for default BMC depth (20)
                    always["result_less_than_4846"](dut.result < 4846)
                else:
                    # this should fail
                    always["result_less_than_4845"](dut.result < 4845)
        
        run_check(Check_BMC, engine)

check_bmc_mode()

check engine=None
  positive test:  PASSED
  negative test:  PASSED
check engine=smtbmc
  positive test:  PASSED
  negative test:  PASSED
check engine=btor btormc
  positive test:  PASSED
  negative test:  PASSED
check engine=btor pono
  positive test:  PASSED
  negative test:  PASSED
check engine=abc bmc3
  positive test:  PASSED
  negative test:  PASSED
check engine=aiger aigbmc
  positive test:  PASSED
  negative test:  PASSED


## engines for prove mode

In [4]:
def check_prove_mode():

    not_working_engines = (
        "aiger avy",     # segfaults
        "aiger rIC3",    # rIC3 not in docker image (maybe get from here https://github.com/gipsyh/rIC3-HWMCC24)
        "aiger suprove"  # suprove not in docker image (maybe get from here https://github.com/sterin/super-prove-build)
    )

    working_engines = (
        None,
        "smtbmc",
        "abc pdr" # slow
    )

    for engine in working_engines:

        class Check_Prove(BaseTestCase, entity=ExampleEntity):
            _yosys_params_ = YosysParams(prove=True, prove_depth=5, engines=engine)

            def formal_properties(self, dut: ExampleEntity):
                
                always["stable_when_not_en"](When(not dut.enable).then_next(stable(dut.result)))
                always["increments_when_en"](When(dut.enable).then_next(dut.result == prev(dut.result) + prev(dut.input)))

                if self.fail_check:
                    always["result_is_not_10000"](dut.result != 10000)
        
        run_check(Check_Prove, engine)

check_prove_mode()

check engine=None
  positive test:  PASSED
  negative test:  PASSED
check engine=smtbmc
  positive test:  PASSED
  negative test:  PASSED
check engine=abc pdr
  positive test:  PASSED
  negative test:  PASSED


## engines for cover mode

In [5]:
def check_cover_mode():

    working_engines = (
        None,
        "smtbmc",
        "btor btormc"
    )

    for engine in working_engines:

        class Check_Cover(BaseTestCase, entity=ExampleEntity):
            _yosys_params_ = YosysParams(cover=True, engines=engine)

            def formal_properties(self, dut: ExampleEntity):
                
                if not self.fail_check:
                    cover["result_is_2345"](dut.result == 2345)
                else:
                    cover["result_is_12345"](dut.result == 12345)
        
        run_check(Check_Cover, engine)

check_cover_mode()

check engine=None
  positive test:  PASSED
  negative test:  PASSED
check engine=smtbmc
  positive test:  PASSED
  negative test:  PASSED
check engine=btor btormc
  positive test:  PASSED
  negative test:  PASSED


## engines for live mode

Yosys only supports the live mode when using the `aiger suprove` solver. I have not managed to build that yet.